In [ ]:
#1 & 3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------------
# Load dataset (correct sheet)
# -------------------------------
# df = pd.read_excel("ToyotaCorolla.xls", sheet_name="data", engine="xlrd")
df = pd.read_csv("ToyotaCorolla.csv")

# -------------------------------
# i. Matrix plot (data visualization)
# -------------------------------
data = df[['Price', 'Age_08_04', 'KM', 'HP']]

# Rename for simplicity
data = data.rename(columns={'Age_08_04': 'Age'})

# Pairplot (matrix plot)
sns.pairplot(data)
plt.show()

# Correlation matrix
print("\nCorrelation:\n", data.corr())


# -------------------------------
# ii. Data Preparation
# -------------------------------

# a. Categorical variables
print("\nCategorical Columns:\n", df[['Fuel_Type', 'Met_Color']].head())

# b. Convert into binary (dummy variables)
# Using drop_first=True automatically applies the N-1 rule!
df_prepared = pd.get_dummies(df, columns=['Fuel_Type', 'Color'], drop_first=True)

# Force pandas to display all columns
pd.set_option('display.max_columns', None)
# c. Show transformed data
# View the new binary columns
print(df_prepared.head())

In [ ]:
#10
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the data
df = pd.read_csv("SouvenirSales.csv")

# Convert the Date column to actual datetime objects and set as index
# Assuming the format in the CSV is 'Jan-95'
df['Date'] = pd.to_datetime(df['Date'], format='%b-%y')
df.set_index('Date', inplace=True)

# i. Create a well-formatted time plot
plt.figure(figsize=(10, 5))
plt.plot(df.index, df['Sales'], color='blue')
plt.title("Monthly Souvenir Sales (1995-2001)")
plt.ylabel("Sales")
plt.xlabel("Year")
plt.show()

# ii. Create a log-scale time plot
plt.figure(figsize=(10, 5))
plt.plot(df.index, np.log(df['Sales']), color='red')
plt.title("Log-Transformed Monthly Souvenir Sales")
plt.ylabel("Log of Sales")
plt.xlabel("Year")
plt.show()

# iv. Partition the data (Last 12 months for validation)
# The validation set is year 2001, the store wants to forecast 2002.
train_df = df.iloc[:-12] # Everything except the last 12 rows
valid_df = df.iloc[-12:] # Only the last 12 rows

print(f"Training set records: {len(train_df)}")
print(f"Validation set records: {len(valid_df)}")

In [ ]:
#2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load data
df = pd.read_csv("ApplianceShipments.csv")

# The 'Quarter' column usually looks like "Q1-1985". Let's split it cleanly.
# If yours is "1985-Q1", change the split order below.
df[['Quarter_Name', 'Year']] = df['Quarter'].str.split('-', expand=True)
df['Year'] = pd.to_numeric(df['Year'])

# i & ii. Well-formatted time plot zoomed into 3500-5000
plt.figure(figsize=(10, 5))
plt.plot(df['Quarter'], df['Shipments'], marker='o', color='b')
plt.ylim(3500, 5000) # Zooming in
plt.title("Appliance Shipments (1985-1989)")
plt.xlabel("Time")
plt.ylabel("Shipments (in millions)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# iii. Create four separate lines for Q1, Q2, Q3, Q4
plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='Year', y='Shipments', hue='Quarter_Name', marker='o')
plt.ylim(3500, 5000)
plt.title("Shipments by Quarter")
plt.xlabel("Year")
plt.ylabel("Shipments")
plt.show()

# iv. Yearly aggregated level
yearly_totals = df.groupby('Year')['Shipments'].sum().reset_index()

plt.figure(figsize=(8, 5))
plt.plot(yearly_totals['Year'], yearly_totals['Shipments'], marker='o', color='green')
plt.title("Total Yearly Appliance Shipments")
plt.xlabel("Year")
plt.ylabel("Total Shipments")
plt.show()


import plotly.express as px

# 1. Load the data
df = pd.read_csv("ApplianceShipments.csv")

# 2. Format as a recognized date! (Answering part v)
# We convert "Q1 1985" into a proper pandas datetime object
# Rearrange "Q1-1985" into "1985Q1"
# We take the last 4 characters (the year) and add the first 2 characters (the quarter)
formatted_quarters = df['Quarter'].str[-4:] + df['Quarter'].str[:2]

# 2. Convert it using PeriodIndex (which is designed specifically for quarters) 
# and translate it to a standard timestamp so Plotly can graph it
df['Date'] = pd.PeriodIndex(formatted_quarters, freq='Q').to_timestamp()

# 3. Create the interactive plot
# When this runs, it opens an interactive HTML graph in your browser
fig = px.line(df, x='Date', y='Shipments', markers=True, 
              title='Interactive Appliance Shipments')

fig.show()

In [ ]:
pip install nbformat

In [ ]:
#4
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
import math

# 1. Load the dataset
df = pd.read_csv("BostonHousing.csv")

# Check how many rows are exact duplicates
print("Duplicate rows:", df.duplicated().sum())

# Check how many unique prices are in the target column
print("Unique prices:", df['MEDV'].nunique())


# Separate features (X) and target variable (y - MEDV)
# We drop MEDV from X, and optionally CAT_MEDV if it exists in your dataset
X = df.drop(columns=['MEDV', 'CAT. MEDV'], errors='ignore') 
y = df['MEDV']

# 2. Partition the data (60% Training, 40% Validation)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, random_state=1)

# 3. Normalize (Scale) the data - CRITICAL FOR k-NN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid) # Notice we only transform validation, not fit!

# i. Perform k-NN for k from 1 to 5
best_k = 1
lowest_rmse = float('inf')

print("--- k-NN Performance ---")
for k in range(1, 6):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    # Predict on validation set
    predictions = knn.predict(X_valid_scaled)
    rmse = math.sqrt(mean_squared_error(y_valid, predictions))
    
    print(f"k = {k} | Validation RMSE: {rmse:.4f}")
    
    # Save the best k (ignoring k=1 due to overfitting)
    if k > 1 and rmse < lowest_rmse:
        lowest_rmse = rmse
        best_k = k

print(f"\nThe best practical k chosen is: {best_k}")

# ii. Predict MEDV for a new tract using the best k
# EXAM DAY: Change these numbers to match the specific tract in your question paper!
new_tract = pd.DataFrame([{
    'CRIM': 0.2, 'ZN': 0.0, 'INDUS': 7.0, 'CHAS': 0, 'NOX': 0.538, 
    'RM': 6.0, 'AGE': 62.0, 'DIS': 4.7, 'RAD': 4, 'TAX': 307, 
    'PTRATIO': 21.0, 'B': 396.9, 'LSTAT': 10.0
}])

# Ensure the new data has the same columns as the training data
new_tract = new_tract[X.columns]

# You MUST scale the new data using the same scaler fitted on the training data
new_tract_scaled = scaler.transform(new_tract)

# Make the final prediction
final_knn = KNeighborsRegressor(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)
predicted_medv = final_knn.predict(new_tract_scaled)

print(f"\nPredicted MEDV for the new tract (k={best_k}): {predicted_medv[0]:.2f}")

In [ ]:
#8
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering

# 1. Load, clean, and set index all in one line
df = pd.read_csv("Universities.csv").dropna().set_index('College Name')

# 2. Isolate numbers and scale them immediately
X = df.select_dtypes(include='number')
X_scaled = StandardScaler().fit_transform(X)

# 3. Draw Dendrogram (calculating linkage directly inside the function)
plt.figure(figsize=(12, 6))
dendrogram(linkage(X_scaled, method='complete'), labels=df.index, leaf_rotation=90)
plt.axhline(7.5, color='r', linestyle='--') 
plt.show()

# 4. Cluster and assign directly to the dataframe
df['Cluster'] = AgglomerativeClustering(n_clusters=3, linkage='complete').fit_predict(X_scaled)

# 5. Print averages
print(df.groupby('Cluster')[X.columns].mean().T)


In [ ]:
#9
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.arima.model import ARIMA

# 1. READ DATA: Load the CSV file and pick only the 'Close' price column
df = pd.read_csv("WalMartStock.csv")
close = df['Close']

# 2. CALCULATE CHANGES: Find the difference between today's price and yesterday's price
# We drop the first row because there is no 'yesterday' for the very first day
diff_series = close.diff().dropna()

# 3. PLOT CHANGES: Create a line graph of these daily changes
# If it looks like random static, it's a "Random Walk"
plt.figure(figsize=(10, 4))
plt.plot(diff_series, color='black')
plt.axhline(0, color='red', linestyle='--') # This line helps us see if changes are centered at zero
plt.title("Day-to-Day Price Changes (Differenced Series)")
plt.show()

# 4. CHECK PATTERNS: Create an Autocorrelation (ACF) plot
# This checks if the current price is mathematically linked to prices from 1, 2, or 3 days ago
plot_acf(close, lags=20)
plt.title("Checking for Patterns (ACF Plot)")
plt.show()

# 5. BUILD THE MODEL: Fit an AR(1) model (Auto-Regressive model with 1 lag)
# Order=(1, 0, 0) tells Python to look back exactly 1 day
model = ARIMA(close, order=(1, 0, 0), trend='c').fit()

# 6. PRINT RESULTS: Show the 'const' (constant) and 'ar.L1' (slope)
# THE VIVA TRICK: If ar.L1 is close to 1.0, the stock is a Random Walk!
print("\n--- Model Results ---")
print(model.params)